# Modélisation — K-Nearest Neighbors (KNN)

Exploration du modèle KNN pour la prédiction de la gravité des accidents de la route (2024).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

## 2. Chargement et Nettoyage des Données

In [ ]:
base_path = Path('..').resolve()
data_path = base_path / 'data' / 'raw' / 'accidents_2024.csv'

df = pd.read_csv(data_path, sep=';', encoding='latin-1')
df.drop_duplicates(inplace=True)
df.dropna(subset=['Gravité (label)'], inplace=True)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f"Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(df['Gravité (label)'].value_counts())

## 3. Encodage, Découpage Train/Test et Normalisation

In [ ]:
y = df['Gravité (label)']
X_raw = df.drop(columns=['Num_Acc', 'Département', 'Gravité (label)'])
X_encoded = pd.get_dummies(X_raw, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train : {X_train_scaled.shape} | Test : {X_test_scaled.shape}")

## 4. Entraînement du Modèle KNN et Évaluation

In [ ]:
# n_neighbors=80 et weights='distance' : choix issu de l'exploration
# (favorise les voisins proches et réduit le bruit des classes minoritaires)
model_knn = KNeighborsClassifier(n_neighbors=80, weights='distance')
model_knn.fit(X_train_scaled, y_train)

y_pred = model_knn.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

## 5. Validation Croisée (5-Fold Stratifié)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_knn, X_train_scaled, y_train,
                          cv=cv, scoring='recall_macro')

print(f"Recall macro — CV 5-fold : {scores.mean():.3f} (+/- {scores.std():.3f})")
print(f"Scores par fold : {scores.round(3)}")

## 6. Matrice de Confusion

In [ ]:
labels = ['Blessé léger', 'Blessé hospitalisé', 'Tué']
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title('Matrice de confusion — KNN (k=80, weights=distance)')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()